# Disaster Tweets: Transformer Ensemble on Kaggle GPUs
**Kaggle Getting Started competition** (`nlp-getting-started`): predict whether a tweet is about a real disaster (`1`) or not (`0`).

- **Metric:** F1 score of the positive class, so the decision threshold matters.
- **Data:** 7,613 labelled and 3,263 test tweets, each with an optional `keyword` and a free-text `location`.
- **Approach**, following the strongest public notebooks (fine-tuned transformers beat every classical model here):
  1. light cleaning that keeps what a tweet model expects (links -> `http`, mentions -> `@user`, broken encodings fixed);
  2. resolve duplicated tweets that were labelled both ways;
  3. 5-fold fine-tuning of two large transformers with different strengths:
     [Twitter-RoBERTa-large](https://huggingface.co/cardiffnlp/twitter-roberta-large-2022-154m) (RoBERTa-large further
     pre-trained on 154M tweets) and [DeBERTa-v3-large](https://huggingface.co/microsoft/deberta-v3-large)
     (the strongest general-purpose encoder), with the keyword passed as a second text segment;
  4. a TF-IDF + logistic-regression model as a cheap, different third opinion;
  5. blend the out-of-fold probabilities and pick the threshold that maximises out-of-fold F1.

### How to run on Kaggle
1. Create a notebook from the competition page (the data is attached as `nlp-getting-started`) and import this file.
2. Settings: **Accelerator GPU T4 x2** (a single P100 also works, the two models then train one after the other),
   **Internet ON** (the models download from Hugging Face).
3. *Save Version -> Save & Run All* (about 1 hour on T4 x2), then submit `submission.csv` from the version's Output tab.

Off Kaggle, or without a GPU, the notebook switches to a **DEBUG** mode that runs the whole pipeline in a few minutes
on 300 tweets with a base-size model, to check the code; its predictions are not meant to be submitted.

The leaderboard's perfect 1.000 scores come from the publicly available source labels; honest models score about 0.83-0.85.

In [ ]:
import html, os, re, subprocess, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from scipy.sparse import hstack

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 140)

SEED = 42
TARGET = 'target'
N_GPUS = torch.cuda.device_count()
DEBUG = N_GPUS == 0
N_FOLDS = 2 if DEBUG else 5

# (Hugging Face name, short tag, learning rate, batch size, epochs)
MODELS = [
    ('cardiffnlp/twitter-roberta-large-2022-154m', 'twroberta_large', 1e-5, 16, 3),
    ('microsoft/deberta-v3-large', 'deberta_v3_large', 1e-5, 16, 3),
]
if DEBUG:
    MODELS = [('cardiffnlp/twitter-roberta-base-2022-154m', 'debug_base', 2e-5, 16, 1)]
print(f'GPUs: {N_GPUS} | DEBUG: {DEBUG} | folds: {N_FOLDS} | models: {[m[1] for m in MODELS]}')

# Plot styling: one fixed colour per class, recessive grid
BLUE, ORANGE = '#2a78d6', '#eb6834'
plt.rcParams.update({
    'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.edgecolor': '#9a9994', 'axes.titleweight': 'bold', 'axes.titlesize': 11, 'font.size': 9,
})

## 1. Load data

In [ ]:
candidates = [Path('/kaggle/input/competitions/nlp-getting-started'), Path('/kaggle/input/nlp-getting-started')]
if Path('/kaggle/input').exists():
    candidates += [p.parent for p in Path('/kaggle/input').glob('**/train.csv')]
candidates.append(Path('.'))   # local run: the competition files sit next to the notebook
DATA = next(c for c in candidates if (c / 'train.csv').exists())
print('data:', DATA.resolve())
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
if DEBUG:
    train = train.sample(300, random_state=SEED).reset_index(drop=True)
    test, sample = test.head(200).copy(), sample.head(200).copy()
print('train', train.shape, '| test', test.shape)
train.sample(5, random_state=SEED)

## 2. EDA
### 2.1 Class balance, missing fields and duplicates

In [ ]:
print(f'Disaster tweets: {train[TARGET].mean():.2%}')
display(pd.DataFrame({'train_missing_%': train[['keyword', 'location']].isna().mean() * 100,
                      'test_missing_%': test[['keyword', 'location']].isna().mean() * 100}).round(2))
dup = train.groupby('text')[TARGET].agg(['size', 'nunique', 'mean'])
conflicts = dup[dup['nunique'] > 1]
print(f'Duplicated texts: {(dup["size"] > 1).sum()} | labelled both ways: {len(conflicts)} '
      f'({conflicts["size"].sum()} rows) | test tweets that also appear in train: {test["text"].isin(train["text"]).sum()}')
display(conflicts.sort_values('size', ascending=False).head(8))

### 2.2 Keywords and length

In [ ]:
kw = train.assign(keyword=train['keyword'].str.replace('%20', ' ')).groupby('keyword')[TARGET].agg(['mean', 'size'])
kw = kw[kw['size'] >= (2 if DEBUG else 20)].sort_values('mean')
show = pd.concat([kw.head(10), kw.tail(10)])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), gridspec_kw={'width_ratios': [1.1, 1]})
axes[0].barh(show.index, show['mean'], color=[ORANGE if v < 0.5 else BLUE for v in show['mean']], height=0.6)
axes[0].axvline(train[TARGET].mean(), color='#9a9994', ls='--', lw=1)
axes[0].set_title('Share of disaster tweets per keyword (10 lowest, 10 highest)'); axes[0].set_xlim(0, 1)

words = train['text'].str.split().str.len()
for label, color, name in [(1, BLUE, 'disaster'), (0, ORANGE, 'not disaster')]:
    axes[1].hist(words[train[TARGET] == label], bins=range(0, 35), histtype='step', linewidth=2,
                 density=True, color=color, label=name)
axes[1].set_title('Words per tweet'); axes[1].set_xlabel('words'); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

**EDA takeaways**
- Classes are mildly imbalanced (43% disaster); F1 rewards getting the positives right, so the threshold is tuned.
- The keyword is a strong prior (`debris`, `wreckage` are nearly always real disasters, `body bags` or `ruin` rarely are),
  so it is passed to the models next to the text. `location` is a third empty and free-form; it is ignored.
- 18 duplicated tweets carry both labels: they are relabelled by majority vote (ties to 0), which removes contradictory
  training signal.
- Disaster tweets are slightly longer and more formal (news style); casual tweets use disaster words figuratively.

## 3. Cleaning, labels and folds
Twitter-RoBERTa was pre-trained with user names replaced by `@user` and links by `http`, so the same is done here.
HTML entities (`&amp;`) are decoded and the mis-decoded UTF-8 fragments the dataset is full of (`\x89Û_`, `Ûª`)
are repaired or dropped. One stratified 5-fold split is shared by every model, so their out-of-fold (OOF)
predictions can be blended.

In [ ]:
MOJIBAKE = {'\x89Ûª': "'", 'Ûª': "'", '\x89Û÷': "'", '\x89Ûò': '-', '\x89ÛÓ': '-', '\x89Û¢': '', '\x89Û_': '',
            '\x89Û': '', 'Û_': '', 'åÈ': '', 'å£': '£', 'åÊ': ' ', 'Ì©': 'e', 'å¨': ''}

def clean(text):
    t = html.unescape(text)
    for bad, good in MOJIBAKE.items():
        t = t.replace(bad, good)
    t = t.encode('ascii', errors='ignore').decode()
    t = re.sub(r'https?://\S+|www\.\S+', 'http', t)
    t = re.sub(r'@\w+', '@user', t)
    return re.sub(r'\s+', ' ', t).strip()

for d in (train, test):
    d['text_clean'] = d['text'].map(clean)
    d['keyword_clean'] = d['keyword'].fillna('none').str.replace('%20', ' ')

# Majority vote for tweets labelled both ways (exact ties -> 0, "not clearly a real disaster")
vote = train.groupby('text_clean')[TARGET].transform('mean')
train['label'] = (vote > 0.5).astype(int)
print('labels changed:', int((train['label'] != train[TARGET]).sum()))
y = train['label'].values

train['fold'] = -1
for k, (_, va_idx) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train, y)):
    train.loc[va_idx, 'fold'] = k
FOLDS = [(np.where(train['fold'] != k)[0], np.where(train['fold'] == k)[0]) for k in range(N_FOLDS)]

# The training script below reads these two files
train[['keyword_clean', 'text_clean', 'label', 'fold']].to_csv('prepared_train.csv', index=False)
test[['keyword_clean', 'text_clean']].to_csv('prepared_test.csv', index=False)
display(train[['text', 'text_clean']].sample(5, random_state=1))

def best_f1(y_true, p):
    ts = np.linspace(0.2, 0.8, 61)
    scores = [f1_score(y_true, p >= t) for t in ts]
    i = int(np.argmax(scores))
    return scores[i], ts[i]

results = {}

## 4. TF-IDF + logistic regression
Word 1-2 grams and character 2-5 grams of the keyword-prefixed text, weighted by TF-IDF (term frequency x inverse
document frequency: words frequent in this tweet but rare overall weigh most). Seconds to train, and it makes
different mistakes from the transformers.

In [ ]:
docs_tr = (train['keyword_clean'] + ' ' + train['text_clean']).str.lower()
docs_te = (test['keyword_clean'] + ' ' + test['text_clean']).str.lower()

oof, pred = np.zeros(len(train)), np.zeros(len(test))
for tr_idx, va_idx in FOLDS:
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True)
    Xtr = hstack([word.fit_transform(docs_tr.iloc[tr_idx]), char.fit_transform(docs_tr.iloc[tr_idx])]).tocsr()
    to_x = lambda d: hstack([word.transform(d), char.transform(d)]).tocsr()
    lr = LogisticRegression(C=2.0, max_iter=2000).fit(Xtr, y[tr_idx])
    oof[va_idx] = lr.predict_proba(to_x(docs_tr.iloc[va_idx]))[:, 1]
    pred += lr.predict_proba(to_x(docs_te))[:, 1] / N_FOLDS
results['tfidf'] = dict(oof=oof, pred=pred)
print(f'TF-IDF + LR: OOF F1 {best_f1(y, oof)[0]:.5f} at threshold {best_f1(y, oof)[1]:.2f}')

## 5. Fine-tuning the transformers
The training code is written to a script and run as one worker process per model. With two GPUs (T4 x2) each worker
gets its own GPU and the two models train at the same time; with one GPU they run one after the other.

Per fold: input `<s> keyword </s></s> tweet </s>`, AdamW (learning rate 1e-5, weight decay 0.01) with linear warm-up
over the first 10% of steps then linear decay, 3 epochs, batches of 16 tweets of similar length (less padding),
fp32 weights with fp16 mixed precision, and gradient clipping at 1.0.

After each epoch the fold is scored; the epoch with the highest validation **AUC** supplies that fold's OOF and test
predictions. AUC measures how well the model ranks disasters above non-disasters without fixing a threshold.
Validation log-loss is a poor choice here: it is usually lowest after the first epoch, because the model then grows
more confident, while F1 keeps improving for another epoch. Choosing the epoch on the validation fold guards against
the occasional unstable large-model run, at the cost of a slightly optimistic OOF score.

In [ ]:
%%writefile train_transformer.py
import argparse, time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, log_loss, roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
ap = argparse.ArgumentParser()
ap.add_argument('--model', required=True)
ap.add_argument('--tag', required=True)
ap.add_argument('--lr', type=float, default=1e-5)
ap.add_argument('--batch', type=int, default=16)
ap.add_argument('--epochs', type=int, default=3)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
amp = dev == 'cuda'
train = pd.read_csv('prepared_train.csv', keep_default_na=False)
test = pd.read_csv('prepared_test.csv', keep_default_na=False)
y, folds = train['label'].values, train['fold'].values
n_folds = folds.max() + 1

tok = AutoTokenizer.from_pretrained(args.model)
def encode(df):
    e = tok(df['keyword_clean'].tolist(), df['text_clean'].tolist(), truncation=True, max_length=128)
    return [{'input_ids': a, 'attention_mask': b} for a, b in zip(e['input_ids'], e['attention_mask'])]
items, items_te = encode(train), encode(test)
lengths = np.array([len(it['input_ids']) for it in items])

def length_batches(idx, rng):
    order = idx[np.argsort(lengths[idx] + rng.uniform(0, 4, len(idx)))]   # sort by length with a little jitter
    batches = [order[i:i + args.batch] for i in range(0, len(order), args.batch)]
    rng.shuffle(batches)
    return batches

@torch.no_grad()
def predict(model, its):
    model.eval()
    order = np.argsort([len(it['input_ids']) for it in its])
    probs = np.zeros(len(its))
    for s in range(0, len(its), 64):
        idx = order[s:s + 64]
        batch = tok.pad([its[i] for i in idx], return_tensors='pt').to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp):
            logits = model(**batch).logits.float()
        probs[idx] = torch.softmax(logits, -1)[:, 1].cpu().numpy()
    return probs

oof, pred = np.zeros(len(train)), np.zeros(len(test))
t_all = time.time()
for k in range(n_folds):
    t0 = time.time()
    torch.manual_seed(args.seed + k)
    rng = np.random.default_rng(args.seed + k)
    tr_idx, va_idx = np.where(folds != k)[0], np.where(folds == k)[0]
    # .float() keeps fp32 master weights for mixed precision: recent transformers load a checkpoint in its stored dtype,
    # and DeBERTa-v3-large is stored in fp16, which the AMP grad scaler cannot train ("Attempting to unscale FP16 gradients")
    model = AutoModelForSequenceClassification.from_pretrained(args.model, num_labels=2).float().to(dev)
    no_decay = ['bias', 'LayerNorm.weight', 'layernorm', 'layer_norm']
    groups = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0},
    ]
    opt = torch.optim.AdamW(groups, lr=args.lr)
    steps = args.epochs * int(np.ceil(len(tr_idx) / args.batch))
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    best = (-np.inf, None, None, 0)
    for epoch in range(args.epochs):
        model.train()
        for idx in length_batches(tr_idx, rng):
            batch = tok.pad([items[i] for i in idx], return_tensors='pt').to(dev)
            labels = torch.tensor(y[idx], device=dev)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp):
                logits = model(**batch).logits
            loss = torch.nn.functional.cross_entropy(logits.float(), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scale = scaler.get_scale()
            scaler.step(opt); scaler.update()
            if scaler.get_scale() >= scale:   # the scaler skips steps whose gradients overflowed; skip the LR step too
                sched.step()
            opt.zero_grad(set_to_none=True)
        p_va = predict(model, [items[i] for i in va_idx])
        auc = roc_auc_score(y[va_idx], p_va)
        ll = log_loss(y[va_idx], np.clip(p_va, 1e-6, 1 - 1e-6))
        f1 = max(f1_score(y[va_idx], p_va >= t) for t in np.linspace(0.2, 0.8, 61))
        print(f'[{args.tag}] fold {k} epoch {epoch + 1}: val AUC {auc:.4f}, log-loss {ll:.4f}, '
              f'best-threshold F1 {f1:.4f} ({time.time() - t0:.0f}s)', flush=True)
        if auc > best[0]:
            best = (auc, p_va, predict(model, items_te), epoch + 1)
    oof[va_idx] = best[1]
    pred += best[2] / n_folds
    print(f'[{args.tag}] fold {k} done: kept epoch {best[3]} ({time.time() - t0:.0f}s)', flush=True)
    del model, opt
    if amp:
        torch.cuda.empty_cache()

np.savez(f'preds_{args.tag}.npz', oof=oof, pred=pred)
print(f'[{args.tag}] finished in {time.time() - t_all:.0f}s', flush=True)

In [ ]:
t0 = time.time()
procs = []
for i, (name, tag, lr, batch, epochs) in enumerate(MODELS):
    env = dict(os.environ)
    if N_GPUS > 0:
        env['CUDA_VISIBLE_DEVICES'] = str(i % N_GPUS)
    cmd = [sys.executable, 'train_transformer.py', '--model', name, '--tag', tag,
           '--lr', str(lr), '--batch', str(batch), '--epochs', str(epochs), '--seed', str(SEED)]
    log = open(f'log_{tag}.txt', 'w')
    procs.append((tag, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT), log))
    if N_GPUS < 2:            # one GPU (or CPU): run the models one after the other
        procs[-1][1].wait()
for tag, proc, log in procs:
    proc.wait(); log.close()
    print(open(f'log_{tag}.txt').read()[-3000:])
    assert proc.returncode == 0, f'{tag} failed, see log_{tag}.txt'
print(f'transformers done in {time.time() - t0:.0f}s')

for name, tag, *_ in MODELS:
    d = np.load(f'preds_{tag}.npz')
    results[tag] = dict(oof=d['oof'], pred=d['pred'])

## 6. Blend and threshold
Weights (step 0.1, summing to 1) and the decision threshold are chosen on the OOF predictions of all 7,613 tweets.

In [ ]:
names = list(results)
summary = pd.DataFrame({n: best_f1(y, results[n]['oof']) for n in names}, index=['OOF F1', 'threshold']).T
display(summary.round(4))
display(pd.DataFrame({n: results[n]['oof'] for n in names}).corr().round(3))

from itertools import product
steps = np.round(np.arange(0, 1.01, 0.1), 1)
grid = [w for w in product(steps, repeat=len(names)) if abs(sum(w) - 1) < 1e-6]

best = (-1, None, None)
for w in grid:
    p = sum(wi * results[n]['oof'] for wi, n in zip(w, names))
    f1, thr = best_f1(y, p)
    if f1 > best[0] + 1e-9:
        best = (f1, dict(zip(names, [float(v) for v in w])), thr)
final_oof_f1, weights, threshold = best
blend_oof = sum(weights[n] * results[n]['oof'] for n in names)
blend_test = sum(weights[n] * results[n]['pred'] for n in names)
print(f'blend weights {weights} | threshold {threshold:.2f} | OOF F1 {final_oof_f1:.5f}')

ts = np.linspace(0.2, 0.8, 61)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(ts, [f1_score(y, blend_oof >= t) for t in ts], color=BLUE)
ax.axvline(threshold, color='#9a9994', ls='--', lw=1)
ax.set_title('OOF F1 of the blend vs decision threshold'); ax.set_xlabel('threshold'); ax.set_ylabel('F1')
plt.tight_layout(); plt.show()

## 7. Submission
On Kaggle: after *Save & Run All*, open the version's **Output** tab and submit `submission.csv`.

In [ ]:
sub = sample[['id']].copy()
assert (sub['id'].values == test['id'].values).all()
sub[TARGET] = (blend_test >= threshold).astype(int)
pd.DataFrame({'id': train['id'], 'label': y, **{f'oof_{n}': results[n]['oof'] for n in names}}).to_csv('oof_predictions.csv', index=False)
if DEBUG:
    print('DEBUG run: submission not written (predictions come from a 300-tweet sample).')
else:
    sub.to_csv('submission.csv', index=False)
    print(sub.shape, '| predicted disaster share:', sub[TARGET].mean().round(4), '| train share:', y.mean().round(4))